In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
project_name = 'TEST-WORKFLOW-EXEC-202601030618-統合管理者'
workflow_name = None
default_result_path = None
close_on_fail = False
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
metadata_project_name_ja = 'wizard-test-project-metadata'
metadata_file_name = 'wizard-test-file.txt'
metadata_file_data_name_ja = 'wizard-test-file-metadata'
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for manager ({idp_name_1})')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for executor ({idp_name_2})')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WORKFLOW-%Y%m%d%H%M')
if workflow_name is None:
    workflow_name = input(prompt='Workflow template name')

# ワークフローの実行（example-wizard）

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー実行（Wizardフォーム / カスタムフォームコンポーネント検証）
- シナリオ名: Wizardタスクの挙動検証（カスタムUI表示、戻る/次へ値保持、alias反映、可視性制御）
- 用意するテストデータ: アカウント(user1: manager/プロジェクト管理者, user2: executor/研究者)、プロジェクト名、ワークフローテンプレート名（example-wizard）
- 事前条件: example-wizard ワークフローテンプレート登録が完了していること
- 備考: user1とuser2が同一ユーザーの場合は単一ユーザーテスト、異なる場合は複数ユーザーテストとなる。承認サイクルは無くexecutor内で完結する。

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

import scripts.metadata_v2025
importlib.reload(scripts.metadata_v2025)
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## manager (user1) のログイン情報を用いてGakuNin RDMにログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、プロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から対象プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Workflow」の行の「有効にする」をクリックする

「Workflowアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    else:
        print('Workflow addon is already enabled for this project')

await run_pw(_step)

## 「確認」をクリックする

「アドオンを構成」のパネル内に「Workflow」の行が追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('#activate-workflow-select')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフローテンプレート」ドロップダウンからテンプレートを選択する

指定したテンプレートが選択されること

In [ ]:
async def _step(page):
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 「有効化」をクリックする

「ワークフロー権限の付与」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator("#activateTokenPermissionModal").locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限を付与して有効化」をクリックする

テンプレートが有効化され、有効化済みテンプレート一覧に表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator(f'#activationsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト名をクリックしてプロジェクトダッシュボードに戻る

プロジェクトダッシュボードが表示され、ワークフローパネルにテンプレートリンクが表示されること

In [ ]:
project_url = None

async def _step(page):
    global project_url
    await page.locator(f'//a[contains(@class, "project-title") and normalize-space(text()) = "{project_name}"]').click()
    await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトする

ログイン画面が表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping logout')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) プロジェクトURL/workflowにアクセスしexecutorとしてログインする

「許可が必要です」画面が表示され、「アクセス権をリクエスト」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping executor login for access request')
        return
    await page.goto(project_url.rstrip('/') + '/workflow')
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//button[text() = "アクセス権をリクエスト"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 「アクセス権をリクエスト」をクリックする

「アクセス権がリクエストされました」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access request')
        return
    await page.locator('//button[text() = "アクセス権をリクエスト"]').click()
    await expect(page.locator('//button[text() = "アクセス権がリクエストされました"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) executorをログアウトしプロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch for access approval')
        return
    await grdm.logout(page, idp_name_2, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 「メンバー」をクリックする

「アクセスのリクエスト」セクションが表示され、executorからのリクエストが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access approval')
        return
    await page.locator('#projectSubnav').get_by_role('link', name='メンバー').click()
    await expect(page.locator('//h3[contains(text(), "アクセスのリクエスト")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.request-accept-button')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 権限を「読込み/書込み」に変更し「追加」をクリックする

リクエストが承認され、「アクセスのリクエスト」セクションからexecutorの行が消えること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access approval action')
        return
    await page.locator('#manageAccessRequests .permissions select').select_option('write')
    await page.locator('.request-accept-button').click()
    await expect(page.locator('#manageAccessRequests .permissions select')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトしプロジェクトURLを開きexecutorとしてログインする

プロジェクトダッシュボードが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch to executor')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータ画面が表示され、「新規メタデータ」ボタンが表示されること。

In [ ]:
async def _step(page):
    await page.locator('#projectSubnav').get_by_role('link', name='メタデータ').click()
    await expect(page.locator('[data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること（スキーマ「公的資金による研究データのメタデータ登録」の行が表示され、「メタデータを作成」ボタンが表示されること）。

In [ ]:
async def _step(page):
    await page.locator('[data-test-new-metadata-button]').click()
    await expect(page.locator('[data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('[data-test-new-report-modal-create-report-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「メタデータを作成」をクリックする

メタデータ編集画面（`/registries/drafts/<id>/metadata`）に遷移し、「プロジェクト名 (日本語)」フィールドが編集可能になること。

In [ ]:
import re

async def _step(page):
    await page.locator('[data-test-new-report-modal-create-report-button]').click()
    await expect(page).to_have_url(re.compile(r'/registries/drafts/.+/metadata'), timeout=transition_timeout)
    form = ProjectMetadataForm(page)
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「プロジェクト名 (日本語)」に値を入力する

テキストフィールドに入力値が反映されること。

In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill('プロジェクト名 (日本語)', metadata_project_name_ja)
    await expect(form.get_locator('プロジェクト名 (日本語)')).to_have_value(metadata_project_name_ja, timeout=transition_timeout)

await run_pw(_step)

## (executor) プロジェクトダッシュボードに戻る

プロジェクトダッシュボードが表示され、ワークフローパネルが表示されること。

In [ ]:
async def _step(page):
    await page.goto(project_url)
    await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

ファイル一覧画面が表示され、対象ストレージの行が表示されること。

In [ ]:
async def _step(page):
    await page.locator('#projectSubnav').get_by_role('link', name='ファイル').click()
    await expect(grdm.get_select_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 対象ストレージにテスト用ファイルをアップロードする

ファイルがアップロードされ、ファイル一覧に表示されること。

In [ ]:
import os

async def _step(page):
    test_file_path = os.path.join(work_dir, metadata_file_name)
    with open(test_file_path, 'w') as wf:
        wf.write('Wizard test file')
    dropzone = grdm.get_select_storage_title_xpath(target_storage_name)
    await grdm.drop_file(page, dropzone, test_file_path)
    await expect(page.locator(f'//*[text() = "{metadata_file_name}"]/../following-sibling::*//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)
    await expect(grdm.get_select_file_title_locator(page, metadata_file_name)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) アップロードしたファイルをクリックする

「メタデータ編集」ボタンが有効になること。

In [ ]:
async def _step(page):
    await grdm.get_select_file_title_locator(page, metadata_file_name).click()
    await expect(page.locator('//*[text() = "メタデータ編集"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「メタデータ編集」をクリックする

メタデータ編集ダイアログが表示され、「メタデータ様式」の select が編集可能になること。

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "メタデータ編集"]').click()
    await expect(page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「メタデータ様式」で「研究成果（公的資金対応）のメタデータ登録」を選択する

「データの名称 (日本語)」フィールドが編集可能になること。

（補足: 表示名は「研究成果（公的資金対応）のメタデータ登録」だが、内部 schema.name は `公的資金による研究データのメタデータ登録` で、example-wizard の `_FILE_METADATA(公的資金による研究データのメタデータ登録)` placeholder と一致する。）

In [ ]:
async def _step(page):
    await page.locator('//label[contains(text(), "メタデータ様式")]/following-sibling::select').select_option('研究成果（公的資金対応）のメタデータ登録')
    form = FileMetadataForm(page)
    await expect(form.get_locator('データの名称 (日本語)')).to_be_editable(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「データの名称 (日本語)」に値を入力する

テキストフィールドに入力値が反映されること。

In [ ]:
async def _step(page):
    form = FileMetadataForm(page)
    await form.fill('データの名称 (日本語)', metadata_file_data_name_ja)
    await expect(form.get_locator('データの名称 (日本語)')).to_have_value(metadata_file_data_name_ja, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「保存」をクリックする

メタデータ編集ダイアログが閉じ、ファイルメタデータが保存されること。

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "保存"]').click()
    form = FileMetadataForm(page)
    await expect(form.get_locator('データの名称 (日本語)')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) プロジェクトダッシュボードに戻る

プロジェクトダッシュボードが表示され、ワークフローパネルが表示されること。


In [ ]:
async def _step(page):
    await page.goto(project_url)
    await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## (executor) ワークフローパネルのテンプレートリンクをクリックする

start formが表示され、特殊プレースホルダ4種（`_PROJECT_METADATA` single/multi-select、`_FILE_METADATA`、`_FILE_SELECTOR`）が固有のUIで描画されていること（plain textareaではないこと）。事前に登録した project metadata / file metadata レコードが ProjectMetadataSelector / FileMetadataSelector のリストに表示されていること。

In [ ]:
async def _step(page):
    await page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]').click()
    await expect(page.locator('button[data-test-workflow-start-button]')).to_be_visible(timeout=transition_timeout)
    # 特殊プレースホルダのラベル（4種）
    await expect(page.locator('label:has-text("Project Metadata (single)")')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('label:has-text("Project Metadata (multi-select)")')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('label:has-text("File Metadata")')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('label:has-text("Files")')).to_be_visible(timeout=transition_timeout)
    # _PROJECT_METADATA: records=1 で `ProjectMetadataSelector__list` が single と multi-select の両方に描画される（合計2）
    await expect(page.locator('div[class*="ProjectMetadataSelector__list"]')).to_have_count(2, timeout=transition_timeout)
    # _FILE_METADATA: records=1 で `FileMetadataSelector__list` が描画される
    await expect(page.locator('div[class*="FileMetadataSelector__list"]')).to_have_count(1, timeout=transition_timeout)
    # _FILE_SELECTOR: Files::Browse が `file-browser-list` を描画（静的クラス）
    await expect(page.locator('div[class*="file-browser"]')).to_have_count(1, timeout=transition_timeout)
    # プレースホルダが固有UIに差し替わっている（textareaフォールバックではない）
    await expect(page.locator('form textarea')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「ワークフローを開始」をクリックする

ワークフローが開始され、タスク一覧にwizardタスクの「開く」ボタンが表示されること。

In [ ]:
async def _step(page):
    await page.locator('button[data-test-workflow-start-button]').click()
    # 起動成功時は executor にタスクが即時割当されて activeTab が 'tasks' へ切り替わり、
    # start form を含む start panel が unmount される（submitSuccess alert は短命で到達しないことがあるので起動完了シグナルにしない）
    await expect(page.locator('button[data-test-workflow-start-button]')).to_have_count(0, timeout=transition_timeout)
    # プロジェクトダッシュボードのワークフローパネルにwizardタスクの「開く」ボタンが表示されることを確認
    await page.goto(project_url)
    await expect(page.locator('#workflow-dashboard .fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## (executor) wizardタスクの「開く」をクリックする

wizardダイアログ（`.workflow-task-dialog`）が表示され、progress sidebarに4ページ分の項目が並び、Page1『Export Settings』が現在ページであること。
Page1には Export Targetのドロップダウン（`select`）と `include_details` のチェックボックスが表示されていること。
wizardダイアログには通常タスクの送信ボタン（`data-test-task-dialog-submit`）が存在しないこと。

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard .fa-pencil').click()
    dlg = page.locator('.workflow-task-dialog')
    await expect(dlg).to_be_visible(timeout=transition_timeout)
    await expect(dlg.locator('.progress-sidebar__item')).to_have_count(4, timeout=transition_timeout)
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Export', timeout=transition_timeout)
    # _EXPORT_TARGET(): label の for= は workflow-field-export_target だが該当 id を持つ要素は存在せず、
    # 同じ form-group 内の div 配下に select が描画される（=カスタム UI、textarea フォールバックではない）
    await expect(dlg.locator('label[for="workflow-field-export_target"]')).to_be_visible(timeout=transition_timeout)
    await expect(dlg.locator('label[for="workflow-field-export_target"]').locator('xpath=../div/select')).to_be_visible(timeout=transition_timeout)
    # include_details は boolean → id 付き checkbox
    await expect(dlg.locator('#workflow-field-include_details')).to_be_visible(timeout=transition_timeout)
    # Page1 の export_target は type=multi-line-text だが、カスタム UI に差し替わっているため textarea は存在しないこと
    await expect(dlg.locator('textarea')).to_have_count(0, timeout=transition_timeout)
    # wizard は通常タスクの submit ボタンを持たない（非 wizard との判別）
    await expect(dlg.locator('[data-test-task-dialog-submit]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (executor) Page1「Export Settings」で「次へ」をクリックする

Page2『File Upload』が現在ページとして表示され、`_FILE_UPLOADER(...)` の固有UI（`div[class*="file-uploader"]` 配下に file-browser）が表示されること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('File Upload', timeout=transition_timeout)
    # _FILE_UPLOADER(...) の固有UI（file-uploader）が描画されていること
    await expect(dlg.locator('div[class*="file-uploader"]')).to_have_count(1, timeout=transition_timeout)
    # Page2 の uploaded_files は type=multi-line-text だがカスタムUIに差し替わっているため textarea は存在しないこと
    await expect(dlg.locator('textarea')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)


## (executor) Page2「File Upload」で「次へ」をクリックする

Page3『Details』が現在ページとして表示され、`note` テキスト入力と `_ARRAY_INPUT(...)`（`div[class*="ArrayInput"]` ＋「追加」ボタン）が表示されること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Details', timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-note')).to_be_visible(timeout=transition_timeout)
    await expect(dlg.locator('div[class*="ArrayInput"]')).to_have_count(1, timeout=transition_timeout)
    await expect(dlg.locator('div[class*="ArrayInput"] button[class*="addButton"]:has-text("追加")')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「note」に値を入力する

入力内容が反映されること。

In [ ]:
note_value_a = 'wizard-test-note-A'

async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('#workflow-field-note').fill(note_value_a)
    await expect(dlg.locator('#workflow-field-note')).to_have_value(note_value_a, timeout=transition_timeout)

await run_pw(_step)

## (executor) Page3「Details」で「次へ」をクリックする

Page4『Confirmation』が現在ページとして表示され、alias（`conf_note` / `conf_export_target`）に source の値が反映された disabled な入力として表示されていること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Confirmation', timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-conf_note')).to_be_disabled(timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-conf_note')).to_have_value(note_value_a, timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-conf_export_target')).to_be_disabled(timeout=transition_timeout)

await run_pw(_step)

## (executor) Page4「Confirmation」で「戻る」をクリックする

Page3『Details』が現在ページとして表示され、`note` の入力値が保持されていること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("戻る")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Details', timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-note')).to_have_value(note_value_a, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「note」の値を更新する

更新内容が反映されること。

In [ ]:
note_value_b = 'wizard-test-note-B-updated'

async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('#workflow-field-note').fill(note_value_b)
    await expect(dlg.locator('#workflow-field-note')).to_have_value(note_value_b, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「次へ」をクリックする

Page4『Confirmation』が表示され、alias（`conf_note`）が更新後の値に反映されていること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Confirmation', timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-conf_note')).to_have_value(note_value_b, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「戻る」でPage1「Export Settings」まで戻る

Page1『Export Settings』が現在ページとして表示されること。progress sidebarに4ページが表示されていること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    # Page4 -> Page3
    await dlg.locator('button:has-text("戻る")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Details', timeout=transition_timeout)
    # Page3 -> Page2
    await dlg.locator('button:has-text("戻る")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('File Upload', timeout=transition_timeout)
    # Page2 -> Page1
    await dlg.locator('button:has-text("戻る")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Export', timeout=transition_timeout)
    await expect(dlg.locator('.progress-sidebar__item')).to_have_count(4, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「include_details」のチェックを外す

可視性制御によりprogress sidebarから『Details』項目が除去され、表示ページ数が3になること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('#workflow-field-include_details').uncheck()
    await expect(dlg.locator('.progress-sidebar__item:has-text("Details")')).to_have_count(0, timeout=transition_timeout)
    await expect(dlg.locator('.progress-sidebar__item')).to_have_count(3, timeout=transition_timeout)

await run_pw(_step)

## (executor) Page1「Export Settings」で「次へ」をクリックする

Page2『File Upload』が現在ページとして表示されること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('File Upload', timeout=transition_timeout)

await run_pw(_step)

## (executor) Page2「File Upload」で「次へ」をクリックする

Detailsページがスキップされて直接Page4『Confirmation』が表示されること。alias（`conf_note`）には非表示ページの入力値（更新後の値）が保持されていること。『送信』ボタンが表示され、『次へ』は非表示であること。

In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("次へ")').click()
    await expect(dlg.locator('.progress-sidebar__item--current')).to_contain_text('Confirmation', timeout=transition_timeout)
    await expect(dlg.locator('#workflow-field-conf_note')).to_have_value(note_value_b, timeout=transition_timeout)
    await expect(dlg.locator('button:has-text("送信")')).to_be_visible(timeout=transition_timeout)
    await expect(dlg.locator('button:has-text("次へ")')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (executor) Page4「Confirmation」で「送信」をクリックする

wizard が送信され、同じダイアログ内に result task（非wizard）が自動表示されること（同一プロセスインスタンス内の次タスクはコントローラが自動で openTask するため）。非wizardの送信ボタン（`data-test-task-dialog-submit`）が表示されること。


In [ ]:
async def _step(page):
    dlg = page.locator('.workflow-task-dialog')
    await dlg.locator('button:has-text("送信")').click()
    # wizard送信後、controller は同プロセスの次タスク(result task=非wizard)を自動openし、同じダイアログが非wizard表示に切替わる
    await expect(dlg.locator('[data-test-task-dialog-submit]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## (executor) result task ダイアログに送信データJSONが埋め込まれていることを確認する

送信データJSONに `"note": "<更新後の値>"` と `"include_details": false` が含まれていること。非表示ページの値が保持されていること、可視性で変更した値が反映されていることを確認する。


In [ ]:
async def _step(page):
    # 前ステップで resultTask のダイアログが自動表示済み。`${_submittedDataSummary}` が本文に展開される。
    # buildSummary の JSON はキーからアンダースコアが除去され、コロン前後にスペースが入る書式で出力される
    # （例: include_details → includedetails, "note" : "value"）
    body = page.locator('.workflow-task-dialog .task-dialog__body')
    await expect(body).to_contain_text(f'"note" : "{note_value_b}"', timeout=transition_timeout)
    await expect(body).to_contain_text('"includedetails" : false', timeout=transition_timeout)

await run_pw(_step)


## (executor) 「送信」をクリックする

ワークフローが完了し、タスク一覧から当該ワークフローの「開く」ボタンが表示されないこと。

In [ ]:
async def _step(page):
    await page.locator('[data-test-task-dialog-submit]').click()
    # ダイアログが閉じ、タスク一覧に「開く」ボタンが表示されないこと
    await expect(page.locator('[data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('[data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)


## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}